# Closing Line Value (CLV) Analysis

CLV is the sports bettor's ground truth. This notebook demonstrates CLV computation,
CLV tracking across dimensions (sport, book, market type, stake tier), and the
relationship between CLV and long-term ROI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from speculator_ev_engine.sports.odds import convert_odds, remove_vig
from speculator_ev_engine.sports.edge import (
    compute_edge, compute_clv, expected_clv,
    CLVTracker, roi_vs_ev_reconciliation,
)

## 1. Single Bet CLV Computation

In [ ]:
# You bet Team A at +150, line closes at +120
clv = compute_clv(open_odds=150, close_odds=120, other_open_outcomes=[-170], other_close_outcomes=[-140])
print(f'CLV: {clv.clv:.4f} ({clv.clv*100:.2f}%)')
print(f'Open implied (vig-free): {clv.open_implied:.4f}')
print(f'Close implied (vig-free): {clv.close_implied:.4f}')
print(f'Positive CLV = you beat the close = good')

## 2. Simulated CLV Distribution

In [ ]:
np.random.seed(42)
n_bets = 500
# Simulate: on average, model beats close by 1.5%
open_probs = np.random.uniform(0.25, 0.75, n_bets)
clv_shifts = np.random.normal(0.015, 0.04, n_bets)
close_probs = np.clip(open_probs - clv_shifts, 0.05, 0.95)

clv_values = open_probs - close_probs

plt.figure(figsize=(10, 6))
plt.hist(clv_values, bins=40, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(clv_values), color='red', linestyle='--', label=f'Mean CLV={np.mean(clv_values):.4f}')
plt.xlabel('CLV')
plt.ylabel('Frequency')
plt.title('Simulated CLV Distribution (500 bets)')
plt.legend()
plt.show()

## 3. CLV Tracker: Flagging Patterns

In [ ]:
tracker = CLVTracker()

# Simulate adding records for different sports and books
sports = ['nfl', 'nba', 'mlb', 'nhl']
books = ['pinnacle', 'draftkings', 'fanduel']
market_types = ['spread', 'moneyline', 'total']

np.random.seed(42)
for i in range(200):
    sport = np.random.choice(sports)
    book = np.random.choice(books)
    mkt = np.random.choice(market_types)
    open_odds = np.random.choice([-110, -105, +100, +105, +110, +120, +150, -130])
    # NFL spreads at Pinnacle show strong CLV; NBA totals at FD show negative CLV
    close_shift = np.random.normal(0, 10)
    if sport == 'nfl' and mkt == 'spread' and book == 'pinnacle':
        close_shift = np.random.normal(-15, 8)  # line moves in our favor
    elif sport == 'nba' and mkt == 'total' and book == 'fanduel':
        close_shift = np.random.normal(10, 8)   # line moves against us
    
    close_odds = open_odds + int(close_shift)
    if close_odds == 0:
        close_odds = -101 if open_odds < 0 else 101
    
    tracker.add_record(sport, book, mkt, 'medium', open_odds, close_odds)

print('=== CLV Summary by Sport ===')
for sport, stats in tracker.summary_by('sport').items():
    print(f'{sport}: mean_clv={stats["mean_clv"]:.4f}, n={int(stats["n_bets"])}, '
          f'positive_pct={stats["positive_clv_pct"]:.2f}')

print('\n=== Flagged Patterns ===')
flags = tracker.flag_patterns(min_samples=10)
for f in flags:
    print(f'  {f["group"]}={f["value"]}: {f["direction"]}, mean_clv={f["mean_clv"]:.4f}')

## 4. ROI vs EV Reconciliation

In [ ]:
# Simulate a season of bets with 3% edge on average
np.random.seed(123)
n = 1000
model_probs = np.random.uniform(0.3, 0.7, n)
odds = np.where(model_probs > 0.5, -110, 110)  # rough approximation
outcomes = (np.random.random(n) < model_probs).astype(float)

recon = roi_vs_ev_reconciliation(
    model_probs=model_probs.astype(np.float64),
    odds_american=odds.astype(np.int64),
    outcomes=outcomes,
)

print(f'Actual ROI:     {recon["actual_roi"]:.4f}')
print(f'Expected EV:    {recon["expected_ev_per_unit"]:.4f}')
print(f'Divergence:     {recon["divergence"]:.4f}')
print(f'N bets:         {int(recon["n_bets"])}')
print('\nIf model is well-calibrated, divergence → 0 as N grows.')